[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CityScope/TransitLOSStudies/blob/main/example_metro_area/example.ipynb)

# Transit level of service for any area

**Quick start:** set the place in *Settings* below, then **Runtime → Run all** (in Jupyter: Run → Run All Cells). The first run installs the packages and takes a few minutes more; a municipality then takes a few minutes, a whole metro area longer.

**To choose more**, run the cells one at a time. Each step has an optional choice, and Run all simply takes the defaults:

| Step | Your choice |
|---|---|
| Area | a place name and type (UN functional area, urban centre or municipality), a shape you draw on a map, or your own file |
| Feeds | which of the downloaded transit feeds to use |
| Service date | the day analysed, with a plot of the service on every date |
| Population | census (WorldPop only when needed) or WorldPop columns too |
| Results | which census columns to keep in the download |

Every input comes from the most local source available: the area's own census and official transit feeds when there are, otherwise global data (WorldPop, Mobility Database, Transitland). The level of service uses the standard scoring functions of the transitLOS package, taken from the transit literature.

In [ ]:
#@title 1. Set up (installs on the first run) { display-mode: "form" }
import subprocess, sys
from pathlib import Path
if not Path("bootstrap.py").is_file():                 # not inside the study folder (Colab): use the clone, making it the first time
    folder = Path("TransitLOSStudies/example_metro_area")
    if not folder.is_dir():
        subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/CityScope/TransitLOSStudies.git"])
    sys.path.insert(0, str(folder.resolve()))
import bootstrap
study = bootstrap.setup()
import ui

## 2. Settings

The only cell you need to edit. In Colab it is a form: change the values and run the cell.

In [ ]:
#@title Settings { display-mode: "form" }
# ----- Area: a place name and type, or your own file, or draw it in step 3
PLACE = "Cambridge, Massachusetts, USA"  #@param {type:"string"}
AREA = "municipality"  #@param ["metro_area", "urban_core", "municipality"]
AOI_FILE = ""  #@param {type:"string"}
DRAW = False  #@param {type:"boolean"}
# AREA: metro_area = UN functional urban area (urban centre + commuting zone), urban_core = UN urban centre, municipality = the city's boundary
# AOI_FILE: a .gpkg, .geojson, .kml or zipped shapefile (its path; "upload" picks one from your computer in Colab). DRAW: draw the area on a map

# ----- When
DAY = "weekday"  #@param ["weekday", "saturday", "sunday"]
START_TIME = "06:00"  #@param {type:"string"}
END_TIME = "22:00"  #@param {type:"string"}

# ----- Data
FEEDS = "download"  #@param {type:"string"}
WORLDPOP = "auto"  #@param ["auto", "include", "never"]
# FEEDS: "download" finds the area's feeds online, or the folder with your GTFS feeds (a Python list of zips, folders and links also works)
# WORLDPOP: auto = WorldPop only where there is no census (a census of small areas replaces it, nothing to download);
#           include = also the WorldPop columns next to a census (a one-time download of tens of GB: WorldPop's age and sex rasters of the whole country); never = no WorldPop columns (without a census it is still used for the population)

## 3. The area

The map shows the area that will be analysed. With `DRAW` ticked, draw a polygon first (the polygon button, top left), then run the next cell.

In [ ]:
#@title Draw the area (only with DRAW) { display-mode: "form" }
if DRAW:
    draw_map = ui.draw_area(study.area_from_place(PLACE, AREA))
    display(draw_map)

In [ ]:
#@title The area { display-mode: "form" }
if DRAW:
    assert draw_map.drawn, "Nothing drawn: draw a polygon on the map above, then run this cell and the ones below again."
    aoi = study.area_from_geojson(draw_map.drawn[-1])
else:
    aoi = study.get_area(PLACE, AREA, ui.upload_file() if AOI_FILE == "upload" else AOI_FILE or None)
ui.show_area(aoi)

## 4. Transit feeds

The feeds are downloaded and listed with their agencies, source, stops in the area and trips per day, all ticked. **Untick the ones you do not want.** Some sources need a free API key (Transitland, Spain's NAP): without it they are skipped and fewer feeds are found. In Colab add the key as a Secret (key icon in the left bar) named `TRANSITLAND_API_KEY` or `NAP_API_KEY` and allow notebook access; on your own machine set the environment variable.

In [ ]:
#@title Download the feeds and choose { display-mode: "form" }
feeds = study.list_feeds(aoi, FEEDS)
feed_choice = ui.select_feeds(feeds)
feed_choice

## 5. Service date

The plot shows the service of the chosen feeds on each date (stops × trips in the area). The suggested date is a typical day of the type set in Settings. **To use another date, set it in the date box.** A date you set is used as it is.

In [ ]:
#@title Suggested service date { display-mode: "form" }
dates = study.service_dates(aoi, feed_choice.value, DAY)
date_choice = ui.pick_date(dates)
date_choice

## 6. Run

This gets the streets and the census, then computes the level of service. With a census of small enough areas the census gives the population and WorldPop is not downloaded.

In [ ]:
#@title Run the study { display-mode: "form" }
zip_path = study.run(aoi, day=DAY, start_time=START_TIME, end_time=END_TIME, feed_names=feed_choice.value, service_date=date_choice.value,
                    worldpop=WORLDPOP)
print(zip_path)

## 7. The results on the map

The map shows the finest H3 hexagons or circles and the finest census level, with the stops and routes. The chart button (bottom left) opens the distribution, regression, ANOVA, discretization and metadata tabs.

In [ ]:
#@title Interactive map { display-mode: "form" }
ui.show_map(zip_path)

## 8. Census columns and download

The results hold the census columns of every source, in groups, all ticked. **Untick a group to leave it out, or press *Show columns* to see each column (name, description, unit, census level, and the result tables that hold it) and tick single ones.** The search box filters the columns. Then run the next cell. The streets, ids, population and level of service are always kept.

The zip holds `streets.parquet` (every street segment with its level of service), `h3_res<N>.parquet` (H3 hexagons at each resolution: population, level of service and the census columns) and `census_<level>.parquet` (the census at its finest level).

In [ ]:
#@title Census column groups { display-mode: "form" }
columns = study.result_columns(aoi)
column_choice = ui.select_columns(columns)
column_choice

In [ ]:
#@title Download the results { display-mode: "form" }
zip_path = study.export_results(aoi, column_choice.value)
ui.download(zip_path)